In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 63.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print("--- Question 1: Label Encoding ---")
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train_df['label'] = train_df['answer'].map(label_map)

label_150 = train_df.loc[150, 'label']
print(f"Encoded numeric label for row at index 150: {label_150}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).


--- Question 1: Label Encoding ---
Encoded numeric label for row at index 150: 2


In [4]:
print("\n--- Question 2: Prompt-Option Formatting ---")
row_0 = train_df.iloc[0]
formatted_input = str(row_0['prompt']) + " [SEP] " + str(row_0['B'])
print(f"Exact character length of formatted input string: {len(formatted_input)}")


--- Question 2: Prompt-Option Formatting ---
Exact character length of formatted input string: 407


In [5]:
print("\n--- Question 3: Single-Row MCQ Tokenization ---")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

prompts = [str(row_0['prompt'])] * 5
options = [str(row_0['A']), str(row_0['B']), str(row_0['C']), str(row_0['D']), str(row_0['E'])]

tokens = tokenizer(
    prompts, 
    options, 
    padding="max_length", 
    truncation=True, 
    max_length=128, 
    return_tensors="pt"
)

input_ids = tokens['input_ids'].unsqueeze(0) 
print(f"Shape of input_ids tensor: {input_ids.shape}")
print(f"Value of the second dimension (num_choices): {input_ids.shape[1]}")


--- Question 3: Single-Row MCQ Tokenization ---


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Shape of input_ids tensor: torch.Size([1, 5, 128])
Value of the second dimension (num_choices): 5


In [6]:
print("\n--- Question 4: Batch MCQ Tokenization ---")
batch_size = 16
num_choices = 5
seq_length = 128
total_positions = batch_size * num_choices * seq_length
print(f"Total token positions for [16, 5, 128] tensor: {total_positions}")


--- Question 4: Batch MCQ Tokenization ---
Total token positions for [16, 5, 128] tensor: 10240


In [7]:
print("\n--- Question 5: Multiple-Choice Logits ---")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased").to(device)
input_ids = input_ids.to(device)
attention_mask = tokens['attention_mask'].unsqueeze(0).to(device)

with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

print(f"Logits shape: {outputs.logits.shape}")
print(f"Number of logits produced for one question: {outputs.logits.shape[1]}")


--- Question 5: Multiple-Choice Logits ---


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([1, 5])
Number of logits produced for one question: 5


In [8]:
print("\n--- Question 6: Supervised Loss Tensor ---")
label_tensor = torch.tensor([row_0['label']]).to(device)

with torch.no_grad():
    outputs_with_loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=label_tensor)

print(f"Dimensions of the loss tensor: {outputs_with_loss.loss.ndim}")


--- Question 6: Supervised Loss Tensor ---
Dimensions of the loss tensor: 0


In [9]:
print("\n--- Question 7: LoRA Trainable Parameters ---")
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS 
)

lora_model = get_peft_model(model, lora_config)
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params}")


--- Question 7: LoRA Trainable Parameters ---
Number of trainable parameters: 295681


In [10]:
print("\n--- Question 8: Hugging Face Dataset Preparation ---")
print("When input_ids has the shape [5, 128] for a single row,")
print("the number of tokenized choices stored in input_ids is 5.")


--- Question 8: Hugging Face Dataset Preparation ---
When input_ids has the shape [5, 128] for a single row,
the number of tokenized choices stored in input_ids is 5.


In [11]:
print("\n--- Question 9: Tiny LoRA Fine-Tuning ---")
subset_df = train_df.head(32).copy()

dataset_list = []
for idx, row in subset_df.iterrows():
    p_list = [str(row['prompt'])] * 5
    o_list = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    
    toks = tokenizer(p_list, o_list, padding="max_length", truncation=True, max_length=64)
    
    dataset_list.append({
        'input_ids': toks['input_ids'],
        'attention_mask': toks['attention_mask'],
        'labels': row['label']
    })

hf_dataset = Dataset.from_list(dataset_list)

training_args = TrainingArguments(
    output_dir="./tiny_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none"
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=hf_dataset,
)

print("Starting tiny fine-tuning...")
trainer.train()
print(f"Final global_step reported by the Trainer: {trainer.state.global_step}")


--- Question 9: Tiny LoRA Fine-Tuning ---
Starting tiny fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.570186
2,1.570080
3,1.605430
4,1.711814


Final global_step reported by the Trainer: 4


In [12]:
print("\n--- Question 10: Probability Assigned to Option E After Fine-Tuning ---")
lora_model.eval()
with torch.no_grad():
    toks_q10 = tokenizer(prompts, options, padding="max_length", truncation=True, max_length=64, return_tensors="pt")
    in_ids = toks_q10['input_ids'].unsqueeze(0).to(device)
    att_mask = toks_q10['attention_mask'].unsqueeze(0).to(device)
    
    final_outputs = lora_model(input_ids=in_ids, attention_mask=att_mask)
    
probabilities = torch.nn.functional.softmax(final_outputs.logits, dim=1)

prob_E = probabilities[0][4].item()
print(f"Probability assigned to Option E: {prob_E:.4f}")


--- Question 10: Probability Assigned to Option E After Fine-Tuning ---
Probability assigned to Option E: 0.1965
